In [1]:
!pip install google-genai --quiet

In [2]:
!pip install python-dotenv --quiet

from dotenv import load_dotenv
import os

load_dotenv("/home/jovyan/work/.env")
api_key = os.getenv("GOOGLE_API_KEY")
print("Key loaded:", api_key is not None)

Key loaded: True


In [3]:
from google import genai

client = genai.Client(api_key=api_key)

In [5]:
def build_prompt_v1(diagnosis, species="bonsai"):
    return f"""You are a bonsai care expert. A customer's {species} tree has been diagnosed with: {diagnosis}.

Give a short, clear rescue plan to fix this problem. Keep it to 3-4 actionable steps."""

diagnosis_example = "overwatering"
prompt_v1 = build_prompt_v1(diagnosis_example)

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=prompt_v1
)
print(response.text)

Here is your quick rescue plan to save an overwatered bonsai:

1. **Pause Watering Immediately**  
   Stop watering right away. Move the tree to a warm spot with bright, indirect sunlight and good airflow to help the soil dry out naturally. 

2. **Inspect and Trim Rotting Roots**  
   Gently lift the tree from its pot. Look for soft, black, or foul-smelling roots (root rot). Carefully prune away any dead, mushy roots using clean, sharp shears to prevent rot from spreading.

3. **Improve Drainage**  
   Ensure the pot’s drainage holes are clear. If the current soil is compacted or holds too much moisture, repot (or "slip-pot") the tree into a well-draining, gritty bonsai soil mix that promotes root oxygenation.

4. **Adopt the "Touch Test" Routine**  
   Moving forward, never water on a fixed schedule. Only water when the top inch of the soil feels dry to the touch, and always let excess water drain completely through the bottom.


In [7]:
import mlflow

mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("bonsai_prompts")

2026/09/30 04:03:18 INFO mlflow.tracking.fluent: Experiment with name 'bonsai_prompts' does not exist. Creating a new experiment.


<Experiment: artifact_location='/mlflow/artifacts/2', creation_time=1790740998330, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1790740998330, lifecycle_stage='active', name='bonsai_prompts', tags={}, trace_location=None, workspace='default'>

In [8]:
with mlflow.start_run(run_name="prompt_v1_direct"):
    mlflow.log_param("prompt_style", "direct_instruction")
    mlflow.log_param("diagnosis", diagnosis_example)
    mlflow.log_param("model", "gemini-3.6-flash")
    mlflow.log_text(prompt_v1, "prompt_template.txt")
    mlflow.log_text(response.text, "response_output.txt")
    print("Logged prompt_v1_direct")

Logged prompt_v1_direct
🏃 View run prompt_v1_direct at: http://mlflow:5000/#/experiments/2/runs/faa3526b0a3f4a558b3f26fc65145359
🧪 View experiment at: http://mlflow:5000/#/experiments/2


In [9]:
def build_prompt_v2(diagnosis, species="bonsai"):
    return f"""You are BonsAI, a warm and reassuring plant care assistant.

A customer's {species} tree shows signs of: {diagnosis}. They may be worried about their plant.

Respond in this exact format:
- One reassuring sentence (don't worry, this is fixable)
- "Steps to fix it:" followed by exactly 3 numbered steps
- One sentence on how to prevent this in future

Keep the tone warm but professional."""

prompt_v2 = build_prompt_v2(diagnosis_example)

response_v2 = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=prompt_v2
)
print(response_v2.text)

Please don't worry, overwatering is a very common issue and your bonsai can easily recover with a little gentle care.

Steps to fix it:
1. Pause watering immediately and allow the soil to dry out significantly.
2. Ensure the drainage holes at the bottom of your pot are clear so excess moisture can escape freely.
3. Trim away any yellowing leaves or mushy roots to help the tree focus its energy on healthy new growth.

To prevent this in the future, always test the soil with your finger and only water your bonsai when the top inch feels dry to the touch.


In [10]:
with mlflow.start_run(run_name="prompt_v2_warm_structured"):
    mlflow.log_param("prompt_style", "warm_structured_format")
    mlflow.log_param("diagnosis", diagnosis_example)
    mlflow.log_param("model", "gemini-3.6-flash")
    mlflow.log_text(prompt_v2, "prompt_template.txt")
    mlflow.log_text(response_v2.text, "response_output.txt")
    print("Logged prompt_v2_warm_structured")

Logged prompt_v2_warm_structured
🏃 View run prompt_v2_warm_structured at: http://mlflow:5000/#/experiments/2/runs/c3defd0578474b41a1d5315766240d1e
🧪 View experiment at: http://mlflow:5000/#/experiments/2
